# Markdown Cleaner Test

This notebook tests the `clean_markdown_for_rag()` function from the markdown cleaner.

In [3]:
import sys
sys.path.insert(0, '..')

from markdown_cleaner import clean_markdown_for_rag
from vector_engine import chunk_text

print('Imports OK')

Imports OK


## Test 1: Basic Markdown Cleanup

In [4]:
sample = """# Project

![badge](https://img.shields.io/badge/Python-3776AB)
<img src="logo.png" width="200"/>

This is **important** content about the project. It uses emojis 🚀 and 😊.

<p align="center"><img src="demo.gif"/></p>

## Features

- Feature 1: does something
- Feature 2: does another thing
"""

print('=== BEFORE ===')
print(repr(sample[:200]))
print()
cleaned = clean_markdown_for_rag(sample)
print('=== AFTER ===')
print(cleaned)

=== BEFORE ===
'# Project\n\n![badge](https://img.shields.io/badge/Python-3776AB)\n<img src="logo.png" width="200"/>\n\nThis is **important** content about the project. It uses emojis 🚀 and 😊.\n\n<p align="center"><img src='

=== AFTER ===
# Project

This is **important** content about the project. It uses emojis  and .

## Features

- Feature 1: does something
- Feature 2: does another thing


## Test 2: Empty / Edge Cases

In [5]:
# Empty string
assert clean_markdown_for_rag('') == ''
print('Empty: OK')

# No markdown, plain text
assert clean_markdown_for_rag('Hello world') == 'Hello world'
print('Plain text: OK')

# Only images
result = clean_markdown_for_rag('![img](url.png)')
print(f'Only images: {repr(result)}')

# Emoji only
result = clean_markdown_for_rag('🚀😊')
print(f'Emoji only: {repr(result)}')
print('Edge cases done')

Empty: OK
Plain text: OK
Only images: ''
Emoji only: ''
Edge cases done


## Test 3: Real GitHub Profile README

In [15]:
import httpx

# Fetch the profile README from GitHub
url = 'https://api.github.com/repos/ridhwanrazaliwork/google-cloud-data-management-project/readme'
resp = httpx.get(url, headers={
    'Accept': 'application/vnd.github.raw+json',
    
})
if resp.status_code == 200:
    raw = resp.text
    print(f'Raw length: {len(raw)} chars')
    cleaned = clean_markdown_for_rag(raw)
    print(f'Cleaned length: {len(cleaned)} chars')
    print(f'Reduction: {len(raw) - len(cleaned)} chars')
    print()
    print('=== CLEANED PREVIEW ===')
    print(cleaned[:5000])
else:
    print('Could not fetch profile README:', resp.status_code)
    print('Using local sample instead...')

Raw length: 17805 chars
Cleaned length: 15654 chars
Reduction: 2151 chars

=== CLEANED PREVIEW ===
# Big Data Analytics Pipeline: Performance Benchmarking with Apache Spark and Apache Hive on Datafiniti Hotel Reviews

> *This project would not have been possible without the collaborative effort, hard work, and dedication of our project team: Ridhwan Razali (Project Lead), Azmal Mukhsin, Amir Shafiq, Hisyam Abdullah, Ahsan Habib, Moy Jia Qin, Liu Yi Qian, and Muhammad Shahir. Thank you all for making this happen.*

## Project Team & Contributions

| Name | Project Role | Primary Focus Areas |
| :--- | :--- | :--- |
| **Ridhwan Razali** | Project Lead / Data Engineer | End-to-end pipeline architecture, Cloud Run ingestion logic |
| **Azmal Mukhsin** | Big Data Engineer | Dataproc cluster configuration, PySpark optimization |
| **Amir Shafiq** | Data Analyst | Data Quality Engineering, Dataproc Jupyter notebooks |
| **Hisyam Abdullah** | Warehousing Specialist | Apache Hive (Tez) external

## Test 4: Chunking with Metadata Injection

In [7]:
sample = """# Project

## Overview
This is a demo project.

## Installation
Run `pip install` to get started.

## Usage
Execute the script.
"""

chunks = chunk_text(sample, 'demo-repo', updated_at='2026-06-15T10:30:00Z')

print(f'Total chunks: {len(chunks)}')
print()
for chunk_id, text, meta in chunks:
    print(f'--- {chunk_id} ---')
    print(text)
    print()

Total chunks: 1

--- demo-repo_0_a256eb9e ---
# Project

## Overview
This is a demo project.

## Installation
Run `pip install` to get started.

## Usage
Execute the script.

[Metadata: Repository=demo-repo, Last Updated=2026-06-15]



## Test 5: Chunking Without updated_at (Fallback)

In [8]:
chunks_no_meta = chunk_text(sample, 'demo-repo')
for chunk_id, text, meta in chunks_no_meta[:1]:
    print(f'--- {chunk_id} ---')
    print(text)
    print()
    print('No metadata suffix (None updated_at)')

--- demo-repo_0_a256eb9e ---
# Project

## Overview
This is a demo project.

## Installation
Run `pip install` to get started.

## Usage
Execute the script.

No metadata suffix (None updated_at)
